In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv('../data/Global_Superstore2.csv', encoding = 'latin-1')
df.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,City,State,...,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Shipping Cost,Order Priority
0,32298,CA-2012-124891,31-07-2012,31-07-2012,Same Day,RH-19495,Rick Hansen,Consumer,New York City,New York,...,TEC-AC-10003033,Technology,Accessories,Plantronics CS510 - Over-the-Head monaural Wir...,2309.650,7,0.0,762.1845,933.57,Critical
1,26341,IN-2013-77878,05-02-2013,07-02-2013,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,FUR-CH-10003950,Furniture,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,0.1,-288.7650,923.63,Critical
2,25330,IN-2013-71249,17-10-2013,18-10-2013,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,TEC-PH-10004664,Technology,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,0.1,919.9710,915.49,Medium
3,13524,ES-2013-1579342,28-01-2013,30-01-2013,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,TEC-PH-10004583,Technology,Phones,"Motorola Smart Phone, Cordless",2892.510,5,0.1,-96.5400,910.16,Medium
4,47221,SG-2013-4320,05-11-2013,06-11-2013,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,TEC-SHA-10000501,Technology,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,0.0,311.5200,903.04,Critical


## Make new columns to create motives for conclusions

In [3]:
df['Cost'] = df['Sales'] - df['Profit']
df['Margin_Pct'] = (df['Profit']/df['Sales'] * 100).round(2)
df['Revenue'] = df['Sales'].round(2)
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst= True)
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Quarter'] = df['Order Date'].dt.quarter

## Stock Keeping Units (SKU) Level Summary

In [4]:
sku = df.groupby(['Product ID', 'Product Name', 'Category', 'Sub-Category']).agg(
    Total_Revenue = ('Sales', 'sum'),
    Total_Cost = ('Cost', 'sum'),
    Total_Profit = ('Profit', 'sum'),
    Total_Units = ('Quantity', 'sum'),
    Avg_Discount = ('Discount', 'mean'),
    Avg_Price = ('Sales', lambda x: (x/df.loc[x.index, 'Quantity']).mean()),
    Transactions = ('Order ID', 'count'),
).reset_index()

In [5]:
sku['Margin_Pct'] = (sku['Total_Profit'] / sku['Total_Revenue'] * 100).round(2)
sku['Revenue_Sharing'] = (sku['Total_Revenue'] / sku['Total_Revenue'].sum() * 100).round(3)

## tail-end flag: bottom 20% of revenue = tail-end products

In [6]:
threshold = sku['Total_Revenue'].quantile(0.20)
sku['Is_Tail'] = sku['Total_Revenue'] <= threshold

## Lets look at it all at once

In [7]:
print(f"Total SKUs:      {len(sku):,}")
print(f"Tail-end SKUs:   {sku['Is_Tail'].sum():,} ({sku['Is_Tail'].mean()*100:.0f}%)")
print(f"Total Revenue:   ${sku['Total_Revenue'].sum():,.0f}")
print(f"Avg Margin:      {sku['Margin_Pct'].mean():.1f}%")
print(f"Negative margin: {(sku['Margin_Pct'] < 0).sum()} SKUs")
print(f"\nCategories: {df['Category'].unique()}")
print(f"Sub-Categories: {df['Sub-Category'].unique()}")
print(f"Segments: {df['Segment'].unique()}")
print(f"Years: {sorted(df['Year'].unique())}")
print(f"\nSample SKU table:")
print(sku[['Product Name','Category','Total_Revenue','Total_Profit',
           'Margin_Pct','Is_Tail']].head(5))

Total SKUs:      10,768
Tail-end SKUs:   2,154 (20%)
Total Revenue:   $12,642,502
Avg Margin:      7.7%
Negative margin: 3027 SKUs

Categories: ['Technology' 'Furniture' 'Office Supplies']
Sub-Categories: ['Accessories' 'Chairs' 'Phones' 'Copiers' 'Tables' 'Binders' 'Supplies'
 'Appliances' 'Machines' 'Bookcases' 'Storage' 'Furnishings' 'Art' 'Paper'
 'Envelopes' 'Fasteners' 'Labels']
Segments: ['Consumer' 'Corporate' 'Home Office']
Years: [np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014)]

Sample SKU table:
                        Product Name   Category  Total_Revenue  Total_Profit  \
0     Advantus Photo Frame, Duo Pack  Furniture        159.120        60.390   
1          Advantus Clock, Erganomic  Furniture        350.070         3.360   
2        Advantus Photo Frame, Black  Furniture        974.832      -651.738   
3  Advantus Stacking Tray, Erganomic  Furniture        124.950         4.200   
4           Advantus Frame, Duo Pack  Furniture        222.360       104

Full transaction data

In [8]:
output_dir = '../data/processed'
os.makedirs(output_dir, exist_ok = True)

In [11]:
df_clean = df[[
    'Order ID', 'Order Date', 'Year', 'Month', 'Quarter',
    'Customer ID', 'Customer Name', 'Segment', 'Ship Mode',
    'Country', 'Region', 'State', 'City',
    'Category', 'Sub-Category', 'Product ID', 'Product Name',
    'Sales', 'Cost', 'Profit', 'Margin_Pct',
    'Quantity', 'Discount', 'Shipping Cost', 'Order Priority'
]].copy()

df_clean.to_csv(f'{output_dir}/transactions.csv', index=False)
print(f" transactions.csv → {len(df_clean):,} rows")

 transactions.csv → 51,290 rows


## sku summary

In [12]:
sku.to_csv(f'{output_dir}/sku_summary.csv', index=False)
print(f" sku_summary.csv → {len(sku):,} rows")

 sku_summary.csv → 10,768 rows


## category summary

In [18]:
cat_summary = df.groupby(['Category', 'Sub-Category']).agg(
    Revenue = ('Sales', 'sum'),
    Cost = ('Cost', 'sum'),
    Profit = ('Profit', 'sum'),
    Units = ('Quantity', 'sum'),
    SKUs = ('Product ID', 'nunique'),
    Avg_Discount = ('Discount', 'mean'),
).reset_index()

cat_summary['Margin_Pct'] = (cat_summary['Profit']/cat_summary['Revenue'] * 100).round(2)
cat_summary['Revenue_Share'] = (cat_summary['Revenue']/cat_summary['Revenue'].sum() * 100).round(2)
cat_summary.to_csv(f'{output_dir}/category_summary.csv', index = False)
print(f"category_summary.csv is {len(cat_summary):,} rows")

category_summary.csv is 17 rows


## regional summary

In [19]:
region_summary = df.groupby(['Region', 'Country', 'Segment']).agg(
    Revenue = ('Sales','sum'),
    Cost = ('Cost','sum'),
    Profit = ('Profit','sum'),
    Units = ('Quantity', 'sum'),
    Orders = ('Order ID', 'nunique'),
    Avg_Discount = ('Discount', 'mean'),
).reset_index()
region_summary['Margin_Pct'] = (region_summary['Profit'] / region_summary['Revenue'] * 100).round(2)
region_summary.to_csv(f'{output_dir}/region_summary.csv', index=False)
print(f"region_summary.csv is {len(region_summary):,} rows")

region_summary.csv is 421 rows


## Monthly trend

In [21]:
trend = df.groupby(['Year', 'Month', 'Category']).agg(
    Revenue = ('Sales',  'sum'),
    Profit  = ('Profit', 'sum'),
    Units   = ('Quantity','sum'),
    Avg_Discount = ('Discount','mean'),
).reset_index()
trend['Margin_Pct'] = (trend['Profit'] / trend['Revenue'] * 100).round(2)
trend['Date'] = pd.to_datetime(
    trend['Year'].astype(str) + '-' + trend['Month'].astype(str) + '-01'
)
trend.to_csv(f'{output_dir}/monthly_trend.csv', index=False)
print(f" monthly_trend.csv is {len(trend):,} rows")

 monthly_trend.csv is 144 rows


## discount impact

In [22]:
discount = df.copy()
discount['Discount_Band'] = pd.cut(
    discount['Discount'],
    bins   = [-0.01, 0, 0.10, 0.20, 0.30, 0.40, 0.50, 1.0],
    labels = ['0%','1-10%','11-20%','21-30%','31-40%','41-50%','>50%']
)
discount_summary = discount.groupby(['Discount_Band','Category']).agg(
    Revenue    = ('Sales',    'sum'),
    Profit     = ('Profit',   'sum'),
    Units      = ('Quantity', 'sum'),
    Num_Orders = ('Order ID', 'count'),
).reset_index()
discount_summary['Margin_Pct'] = (
    discount_summary['Profit'] / discount_summary['Revenue'] * 100
).round(2)
discount_summary.to_csv(f'{output_dir}/discount_impact.csv', index=False)
print(f" discount_impact.csv is {len(discount_summary):,} rows")

 discount_impact.csv is 21 rows


/var/folders/h8/51mqc9v517x00_mhcb5s7ncw0000gn/T/ipykernel_59631/1206783225.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  discount_summary = discount.groupby(['Discount_Band','Category']).agg(


## confirming my sanity

In [23]:
print("\n All datasets saved to ../data/processed/")
print("\nKey insights to highlight in the dashboard:")
print(f"  • {(sku['Margin_Pct'] < 0).sum():,} SKUs with negative margin ({(sku['Margin_Pct'] < 0).mean()*100:.0f}% of portfolio)")
print(f"  • {sku['Is_Tail'].sum():,} tail-end SKUs generating only {sku[sku['Is_Tail']]['Total_Revenue'].sum()/sku['Total_Revenue'].sum()*100:.1f}% of revenue")
print(f"  • Avg margin only {sku['Margin_Pct'].mean():.1f}% — significant room for improvement")
best_cat = cat_summary.nlargest(1,'Margin_Pct').iloc[0]
worst_cat = cat_summary.nsmallest(1,'Margin_Pct').iloc[0]
print(f"  • Best sub-category:  {best_cat['Sub-Category']} ({best_cat['Margin_Pct']:.1f}% margin)")
print(f"  • Worst sub-category: {worst_cat['Sub-Category']} ({worst_cat['Margin_Pct']:.1f}% margin)")


 All datasets saved to ../data/processed/

Key insights to highlight in the dashboard:
  • 3,027 SKUs with negative margin (28% of portfolio)
  • 2,154 tail-end SKUs generating only 1.1% of revenue
  • Avg margin only 7.7% — significant room for improvement
  • Best sub-category:  Paper (24.2% margin)
  • Worst sub-category: Tables (-8.5% margin)
